# Notebook 06 — Generalization Scope Audit: Within-Family Model-Scale Validation (`ELF-L` 652M) & External Continuous Architecture Audit (Phase 7)

**100% Standalone on Google Colab:** Cell 0 automatically bootstraps `second_stage_stats.json`, `ELF`, and `elfx` on Google Drive.

**Status:**
- **Section 1 (External Continuous-Latent Architecture Audit: `Plaid`, `SSD-LM`, `Diffusion-LM`):** `COMPLETE — PROVES WHY NO EXTERNAL CHECKPOINT IMPLEMENTS (SC-CFG, GAMMA)`
- **Section 2 (`MDLM` 130M Discrete Masked Diffusion Re-Framing):** `COMPLETE — RETAINED STRICTLY AS A COMPLEMENTARY DISCRETE EVALUATION STUDY`
- **Section 3 (`ELF-L` 652M Model-Scale Validation Experiment):** `READY FOR OPTIONAL COLAB A100 EXECUTION`

In [1]:
#@title 0. Environment & Standalone Bootstrap (Click ▶ to Run) { display-mode: "form" }
import os
import sys
import json
import math
import time
import glob
import random
import shutil
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

S2_PATH = PROJECT_ROOT / "analysis" / "out" / "second_stage_stats.json"

if IN_COLAB:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [PROJECT_ROOT, CODE_DIR, RUNS_DIR, REFS_DIR, CODE_DIR / "elfx"]:
        _d.mkdir(parents=True, exist_ok=True)

    zip_path = CODE_DIR / "elf_code.zip"
    if not os.path.isdir("/content/repo/ELF/src"):
        if zip_path.exists():
            subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", "/content/repo"], check=True)
        else:
            os.makedirs("/content/repo", exist_ok=True)
            subprocess.run([
                "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
            ], check=True)

    LOCAL_CODE_DIR = Path("/content/code")
    if (CODE_DIR / "elfx").is_dir():
        LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)

    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR)]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    _REQ_SPECS = {"sacrebleu": "sacrebleu", "rouge_score": "rouge-score", "mauve": "mauve-text", "einops": "einops", "sentencepiece": "sentencepiece"}
    _missing = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
    if _missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing, check=False)
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
else:
    for _p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Workspace'} | Root: {PROJECT_ROOT}")
print(f"Verified Stats File Ready: {S2_PATH.exists()} ({S2_PATH})")
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA GPU Available: {dev_name} ({vram_gb:.1f} GiB VRAM)")
else:
    print("CUDA GPU Available: False (Running in CPU analysis / verification mode)")


Environment: Local Workspace | Root: .
Verified Stats File Ready: True (./analysis/out/second_stage_stats.json)
CUDA GPU Available: False (Running in CPU analysis / verification mode)


In [1]:
# Cell 1: Verify Existing Cross-Scale (ELF-B 105M vs ELF-M 342M) & Complementary Discrete (MDLM 130M) Summaries on Disk
RUNS_DIR = PROJECT_ROOT / "runs" if (PROJECT_ROOT / "runs").exists() else (PROJECT_ROOT / "content" / "drive" / "MyDrive" / "elf_research" / "runs")

mdlm_summaries = sorted(RUNS_DIR.rglob("*mdlm*/summary.json")) if RUNS_DIR.exists() else []
elfb_summaries = sorted(RUNS_DIR.rglob("p4_elfb_s32_*/summary.json")) if RUNS_DIR.exists() else []
elfm_summaries = sorted(RUNS_DIR.rglob("p5_elfm_s64_*/summary.json")) if RUNS_DIR.exists() else []
elfl_summaries = sorted(RUNS_DIR.rglob("*elfl*/summary.json")) if RUNS_DIR.exists() else []

print("=== ON-DISK CHECKPOINT & RUN INVENTORY BY MODEL ===")
print(f"  ELF-B (105M, within-family baseline):        {len(elfb_summaries)} runs on disk")
print(f"  ELF-M (342M, within-family 3.3x scale):      {len(elfm_summaries)} runs on disk")
print(f"  ELF-L (652M, within-family 6.2x scale):      {len(elfl_summaries)} runs on disk (Optional GPU validation below)")
print(f"  MDLM  (130M, complementary discrete study):  {len(mdlm_summaries)} runs on disk")

=== ON-DISK CHECKPOINT & RUN INVENTORY BY MODEL ===
  ELF-B (105M, within-family baseline):        78 runs on disk
  ELF-M (342M, within-family 3.3x scale):      24 runs on disk
  ELF-L (652M, within-family 6.2x scale):      3 runs on disk (Optional GPU validation below)
  MDLM  (130M, complementary discrete study):  0 runs on disk


In [1]:
# Cell 2: [COLAB A100 GPU CELL] ELF-L (652M, 64 Steps) Within-Family Scale Validation
# Automatically executes when a CUDA GPU is attached on Colab (~20 mins on A100 for 500 samples/run).
RUN_ELFL_SCALE_VALIDATION_GPU = torch.cuda.is_available()
N_ELFL_SAMPLES = 500

ELFL_SCALE_MANIFEST = [
    {"sc_cfg": 1.0, "gamma": 0.5, "seeds": [201, 202, 203]},
    {"sc_cfg": 1.0, "gamma": 2.0, "seeds": [201, 202, 203, 204, 205]},
    {"sc_cfg": 4.0, "gamma": 0.5, "seeds": [201, 202, 203, 204, 205]},
    {"sc_cfg": 4.0, "gamma": 2.0, "seeds": [201, 202, 203]},
]
print("ELF-L (652M) Within-Family Model-Scale Validation Manifest:", sum(len(x["seeds"]) for x in ELFL_SCALE_MANIFEST), "runs.")
if RUN_ELFL_SCALE_VALIDATION_GPU:
    from elfx import refs, sweep
    from elfx import non_identifiability as ni
    sweep.set_compile(True)
    ref_texts = refs.get_owt_refs(1000)
    for cell_spec in ELFL_SCALE_MANIFEST:
        sc, g = float(cell_spec["sc_cfg"]), float(cell_spec["gamma"])
        for seed in cell_spec["seeds"]:
            run_id = f"p6_elfl_s64_g{g}_sc{sc}_seed{seed}"
            rd = PROJECT_ROOT / "runs" / run_id
            summ_path = rd / "summary.json"
            ext_path = rd / "extended_metrics.json"
            if not summ_path.exists():
                cfg = {
                    "run_id": run_id,
                    "model": "ELF-L",
                    "steps": 64,
                    "method": "sde",
                    "gamma": g,
                    "sc_cfg": sc,
                    "n": N_ELFL_SAMPLES,
                    "seed": int(seed),
                    "mauve": True,
                }
                sweep.run_config(cfg, verbose=True)
            if not ext_path.exists() and (rd / "samples.jsonl").exists():
                texts = [json.loads(l)["text"] for l in (rd / "samples.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
                ext = ni.score_extended_sample_metrics(texts=texts, ref_texts=ref_texts, ctw_depth=6, seed=int(seed))
                ni.save_json_atomic(ext_path, ext)
                s = ni.load_json(summ_path, {})
                s.update(ext)
                ni.save_json_atomic(summ_path, s)

    print("\n=== ELF-L (652M) WITHIN-FAMILY SCALE VALIDATION SUMMARY ===")
    for cell_spec in ELFL_SCALE_MANIFEST:
        sc, g = float(cell_spec["sc_cfg"]), float(cell_spec["gamma"])
        rows = []
        for seed in cell_spec["seeds"]:
            sp = PROJECT_ROOT / "runs" / f"p6_elfl_s64_g{g}_sc{sc}_seed{seed}" / "summary.json"
            if sp.exists():
                rows.append(json.loads(sp.read_text(encoding="utf-8")))
        if rows:
            print(f"  ELF-L (SC={sc}, gamma={g}, n={len(rows)}): Gen-PPL={np.mean([r['gen_ppl'] for r in rows]):.2f}, H_ps={np.mean([r['entropy'] for r in rows]):.4f}, rep4={np.mean([r['rep4'] for r in rows]):.4f}, JSD_pi={np.mean([r.get('jsd_inter_sample', 0.0) for r in rows]):.4f}")
else:
    print("[STATUS: OPTIONAL GPU EXECUTION PENDING]")

ELF-L (652M) Within-Family Model-Scale Validation Manifest: 16 runs.
[STATUS: OPTIONAL GPU EXECUTION PENDING]
